# Diffusion Model: Mask-Conditioned DDPM for Diseased Leaf Generation (Linux / Cloud GPU)

Trains a **Denoising Diffusion Probabilistic Model (DDPM)** that generates realistic
GLS-diseased leaf images conditioned on a **binary full-leaf silhouette mask**.

## Architecture
- **TimeEmbeddedUNet** — the existing 4-channel UNet backbone
  (`src/models/unet_inpaint.py`) extended with **sinusoidal time embeddings** and
  **FiLM (Feature-wise Linear Modulation)** conditioning in each block.
- State-dict key structure of `TimeFiLMBlock` matches the Phase-1 `UNetBlock`
  (`conv.{0,1,3,4}`) so the backbone weights warm-start directly from the GAN-A
  Phase-1 checkpoint — no channel adaptation needed.
- **Input**: `[3ch noisy diseased image + 1ch binary leaf mask]` = 4 channels (same as Phase 1)
- **Output**: predicted noise ε ∈ ℝ³ (unbounded — **no sigmoid**)

## Training
- **Objective**: ε-prediction MSE  (noise prediction, Eq. 14 Ho et al. 2020)
- **Schedule**: cosine noise schedule (Nichol & Dhariwal 2021), T=1000
- **Optimizer**: AdamW + CosineAnnealingLR
- **Mixed precision**: AMP fp16 with gradient clipping (max_norm=1.0)
- **Image normalisation**: **[-1, 1]** (DDPM standard — different from GAN-A's [0, 1])
- **Mask stays static** throughout reverse diffusion (it is the conditioning signal)

## Inference
- **DDIM** (Song et al. 2020) — 50 deterministic steps (20× speedup over 1000-step DDPM)

## Data
- **Diseased images**: `data/diseased/gls/` (263 images)
- **Leaf masks**: `data/masks/leaf/gls/` (generated by `diffusion_data_setup.ipynb`)

**Target**: A100 80GB on Lambda Labs cloud

In [1]:
# 1. Imports
import os
import sys
import json
import time
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm import tqdm
import numpy as np
import matplotlib

# Use inline backend in notebooks; Agg for headless cloud runs
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
# 2. Resolve repo root
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == 'gan':
    repo_root = repo_root.parents[1]
elif repo_root.name == 'src':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Repo root: {repo_root}')
# Reference model file (not imported directly — TimeEmbeddedUNet is defined below)
print(f'Reference model: {repo_root / "src" / "models" / "unet_inpaint.py"}')

Repo root: C:\Users\henry\Desktop\isef2026
Reference model: C:\Users\henry\Desktop\isef2026\src\models\unet_inpaint.py


In [3]:
# =============================================================
# 3. CONFIGURATION — edit these to switch datasets / hyperparams
# =============================================================

# ----- Data paths -----
DISEASED_DIR  = os.path.join(str(repo_root), 'data', 'diseased', 'gls')
LEAF_MASK_DIR = os.path.join(str(repo_root), 'data', 'masks', 'leaf', 'gls')

# ----- Output directories -----
CHECKPOINT_DIR = os.path.join(str(repo_root), 'src', 'models', 'checkpoints')
LOG_DIR        = os.path.join(str(repo_root), 'results', 'logs')
SAMPLE_DIR     = os.path.join(str(repo_root), 'results', 'figures', 'diffusion_samples')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,        exist_ok=True)
os.makedirs(SAMPLE_DIR,     exist_ok=True)

# ----- Warm-start from GAN-A Phase 1 (4-channel, leaf-domain pretrained) -----
# Set to None to train from scratch.
WARMSTART_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'full_ganA_epoch150_final.pth')

# ----- Resume DDPM training (set to None to start fresh) -----
RESUME_CHECKPOINT = None  # e.g. os.path.join(CHECKPOINT_DIR, 'diffusion_epoch50.pth')

# ----- Diffusion schedule -----
T_STEPS  = 1000       # total diffusion timesteps
SCHEDULE = 'cosine'   # 'cosine' (Nichol & Dhariwal 2021) or 'linear'

# ----- Training hyperparameters (tuned for A100 80GB) -----
BATCH_SIZE   = 16     # 512x512 fp16 — A100 80GB comfortably fits 16
NUM_EPOCHS   = 500    # DDPM converges slower than GAN; 500 ≈ 3.5h on A100
IMAGE_SIZE   = 512
SAVE_EVERY   = 50     # save checkpoint every N epochs
SAMPLE_EVERY = 25     # save visual samples every N epochs
NUM_WORKERS  = 4
TRAIN_SPLIT  = 0.9    # 236 train / 27 test
SPLIT_SEED   = 42

# ----- Optimizer -----
LEARNING_RATE = 1e-4   # AdamW; lower than GAN's 2e-4 for DDPM stability
WEIGHT_DECAY  = 1e-4

# ----- Model architecture -----
FEATURES       = [64, 128, 256, 512]  # UNet feature widths — match Phase-1 exactly
TIME_EMBED_DIM = 256                  # sinusoidal embedding dimension
IN_CHANNELS    = 4    # 3ch noisy image + 1ch binary leaf mask
OUT_CHANNELS   = 3    # predicted noise ε (3ch)

# ----- Augmentation -----
AUG_CROP_SCALE   = (0.80, 1.0)   # random resized crop scale range
AUG_COLOR_JITTER = dict(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03)

# ----- DDIM inference -----
DDIM_STEPS = 50    # denoising steps at inference (vs 1000 training steps)
DDIM_ETA   = 0.0   # 0.0 = deterministic DDIM; 1.0 = stochastic (DDPM)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Image size: {IMAGE_SIZE}x{IMAGE_SIZE}  |  Batch size: {BATCH_SIZE}')
print(f'Epochs: {NUM_EPOCHS}  |  T: {T_STEPS}  |  Schedule: {SCHEDULE}')
print(f'DDIM: {DDIM_STEPS} steps, eta={DDIM_ETA}')
print(f'Warm-start: {os.path.basename(WARMSTART_CHECKPOINT) if WARMSTART_CHECKPOINT else "None"}')

Device: cuda
GPU: NVIDIA GeForce RTX 4060
GPU memory: 8.6 GB
Image size: 512x512  |  Batch size: 16
Epochs: 500  |  T: 1000  |  Schedule: cosine
DDIM: 50 steps, eta=0.0
Warm-start: full_ganA_epoch150_final.pth


In [4]:
# 4. DDPM noise schedule utilities

def cosine_beta_schedule(T, s=0.008):
    """Cosine noise schedule from Nichol & Dhariwal (2021).
    Avoids the abrupt full-signal destruction of the linear schedule at small t.
    """
    steps = T + 1
    x = torch.linspace(0, T, steps)
    alphas_cumprod = torch.cos(((x / T) + s) / (1.0 + s) * math.pi / 2.0) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1.0 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return betas.clamp(max=0.999)


def build_schedule(T, schedule='cosine', device='cpu'):
    """Precompute and return all schedule-derived tensors on device."""
    if schedule == 'cosine':
        betas = cosine_beta_schedule(T)
    else:
        betas = torch.linspace(1e-4, 0.02, T)   # linear fallback

    alphas         = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

    return {
        'betas':                         betas.to(device),
        'alphas_cumprod':                alphas_cumprod.to(device),
        'alphas_cumprod_prev':           alphas_cumprod_prev.to(device),
        'sqrt_alphas_cumprod':           alphas_cumprod.sqrt().to(device),
        'sqrt_one_minus_alphas_cumprod': (1.0 - alphas_cumprod).sqrt().to(device),
    }


def q_sample(x0, t, sched, noise=None):
    """Forward diffusion: sample x_t from x_0 at timestep t.

    x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * epsilon

    Returns:
        (x_t, epsilon)  — both [B, C, H, W]
    """
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_alpha  = sched['sqrt_alphas_cumprod'][t].view(-1, 1, 1, 1)
    sqrt_1minus = sched['sqrt_one_minus_alphas_cumprod'][t].view(-1, 1, 1, 1)
    return sqrt_alpha * x0 + sqrt_1minus * noise, noise


print('DDPM schedule utilities defined.')

DDPM schedule utilities defined.


In [5]:
# 5. TimeEmbeddedUNet — mask-conditioned DDPM backbone
#
# Architecture mirrors src/models/unet_inpaint.py with two additions:
#   1. SinusoidalPositionEmbedding + MLP projects timestep t → time_embed vector
#   2. TimeFiLMBlock injects time_embed via FiLM after the first Conv-BN in each block
#
# KEY: TimeFiLMBlock.conv sequential uses the SAME key indices {0,1,3,4} as UNetBlock.conv,
# so ALL encoder/decoder/bottleneck weights load directly from the Phase-1 checkpoint
# via strict=False. Only time_embed.* and *.time_proj.* are new (randomly initialized).

class SinusoidalPositionEmbedding(nn.Module):
    """Sinusoidal timestep embeddings (Vaswani et al. 2017).
    Maps integer timesteps t → [B, dim] float embedding.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half   = self.dim // 2
        freqs  = torch.exp(
            -math.log(10000) *
            torch.arange(half, device=device, dtype=torch.float32) / (half - 1)
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)  # [B, half]
        return torch.cat([args.sin(), args.cos()], dim=-1)  # [B, dim]


class TimeFiLMBlock(nn.Module):
    """Double-conv block with FiLM time conditioning.

    State-dict key structure (compatible with Phase-1 UNetBlock for warm-start):
      conv.0.weight / conv.0.bias       ← first Conv2d
      conv.1.{weight,bias,running_*}    ← first BatchNorm2d
      conv.3.weight / conv.3.bias       ← second Conv2d
      conv.4.{weight,bias,running_*}    ← second BatchNorm2d
      time_proj.*                       ← NEW (not in Phase-1, skipped by strict=False)

    Note: conv.2 and the implicit final activation are SiLU (no parameters — same
    positional index as Phase-1's ReLU at conv.2; state_dict entries identical).
    """
    def __init__(self, in_ch, out_ch, time_embed_dim):
        super().__init__()
        # Preserve key layout to match UNetBlock.conv indices
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),   # conv.0
            nn.BatchNorm2d(out_ch),                   # conv.1
            nn.SiLU(),                                 # conv.2 (no params)
            nn.Conv2d(out_ch, out_ch, 3, padding=1),  # conv.3
            nn.BatchNorm2d(out_ch),                   # conv.4
            # No trailing activation here — applied after FiLM below
        )
        # FiLM projection: time_embed → (gamma, beta) pair for out_ch channels
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_embed_dim, out_ch * 2),
        )

    def forward(self, x, t_emb):
        # First Conv-BN-SiLU
        h = self.conv[2](self.conv[1](self.conv[0](x)))
        # FiLM conditioning: scale and shift feature maps by time embedding
        film  = self.time_proj(t_emb)                       # [B, out_ch * 2]
        gamma, beta = film.chunk(2, dim=-1)                 # each [B, out_ch]
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)           # [B, out_ch, 1, 1]
        beta  = beta.unsqueeze(-1).unsqueeze(-1)
        h     = h * (1.0 + gamma) + beta
        # Second Conv-BN-SiLU
        h = F.silu(self.conv[4](self.conv[3](h)))
        return h


class TimeEmbeddedUNet(nn.Module):
    """Mask-conditioned DDPM UNet backbone.

    Input:  x [B, 4, H, W]  noisy image (3ch) concatenated with binary mask (1ch)
            t [B]           integer diffusion timestep
    Output: e [B, 3, H, W]  predicted noise epsilon (unbounded — NO sigmoid)
    """
    def __init__(self, in_channels=4, out_channels=3,
                 features=None, time_embed_dim=256):
        super().__init__()
        if features is None:
            features = [64, 128, 256, 512]

        # --- Time embedding MLP ---
        # SinusoidalPositionEmbedding → Linear(dim, 4*dim) → SiLU → Linear(4*dim, dim)
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_embed_dim),
            nn.Linear(time_embed_dim, time_embed_dim * 4),
            nn.SiLU(),
            nn.Linear(time_embed_dim * 4, time_embed_dim),
        )

        self.pool = nn.MaxPool2d(2)

        # --- Encoder ---
        self.downs = nn.ModuleList()
        chs = in_channels
        for feat in features:
            self.downs.append(TimeFiLMBlock(chs, feat, time_embed_dim))
            chs = feat

        # --- Bottleneck ---
        self.bottleneck = TimeFiLMBlock(features[-1], features[-1] * 2, time_embed_dim)
        chs = features[-1] * 2

        # --- Decoder ---
        self.ups = nn.ModuleList()
        for feat in reversed(features):
            self.ups.append(nn.ConvTranspose2d(chs, feat, 2, stride=2))
            self.ups.append(TimeFiLMBlock(feat * 2, feat, time_embed_dim))
            chs = feat

        # --- Final 1x1 conv (NO sigmoid — output is unbounded epsilon) ---
        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x, t):
        """
        Args:
            x: [B, 4, H, W]  (noisy 3ch image + 1ch mask)
            t: [B]           integer timesteps
        Returns:
            [B, 3, H, W]    predicted noise epsilon
        """
        t_emb = self.time_embed(t)   # [B, time_embed_dim]

        # Encoder with skip connections
        skips = []
        for down in self.downs:
            x = down(x, t_emb)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x, t_emb)

        # Decoder
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x    = self.ups[i](x)          # ConvTranspose2d (upsample)
            skip = skips[i // 2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i + 1](x, t_emb) # TimeFiLMBlock

        return self.final(x)


print('TimeEmbeddedUNet defined.')
print('  conv.{0,1,3,4} keys match Phase-1 UNetBlock — warm-start compatible.')
print('  Final layer: NO sigmoid (predicts unbounded epsilon).')

TimeEmbeddedUNet defined.
  conv.{0,1,3,4} keys match Phase-1 UNetBlock — warm-start compatible.
  Final layer: NO sigmoid (predicts unbounded epsilon).


In [ ]:
# 6. Dataset: paired (GLS diseased image, binary leaf mask)

class LeafDiffusionDataset(Dataset):
    """Loads paired (GLS image, binary leaf mask) for DDPM training.

    Image normalization : [-1, 1]  — DDPM standard (NOT [0,1] like GAN-A).
    Mask                : {0, 1}   — static conditioning; never noised.
    Augmentation        : synchronized geometric transforms on both;
                          color jitter on image only.

    Pairing: both directories are sorted independently; index i in images
    is paired with index i in masks. Run diffusion_data_setup.ipynb first
    to ensure masks are generated in the same order as the source images.
    """
    EXTS = ('.png', '.jpg', '.jpeg')

    def __init__(self, image_dir, mask_dir, image_size=512, split='train',
                 train_fraction=0.9, seed=42, augment=True):
        img_files  = sorted(f for f in os.listdir(image_dir)
                            if f.lower().endswith(self.EXTS))
        mask_files = sorted(f for f in os.listdir(mask_dir)
                            if f.lower().endswith(self.EXTS))

        assert len(img_files)  > 0, f'No images in {image_dir}'
        assert len(mask_files) > 0, f'No masks in {mask_dir}'
        assert len(img_files) == len(mask_files), (
            f'Image/mask count mismatch: {len(img_files)} vs {len(mask_files)}. '
            f'Run diffusion_data_setup.ipynb first.')

        rng     = random.Random(seed)
        indices = list(range(len(img_files)))
        rng.shuffle(indices)
        n_train = int(len(indices) * train_fraction)
        idx     = indices[:n_train] if split == 'train' else indices[n_train:]

        self.image_files  = [img_files[i]  for i in idx]
        self.mask_files   = [mask_files[i] for i in idx]
        self.image_dir    = image_dir
        self.mask_dir     = mask_dir
        self.image_size   = image_size
        self.augment      = augment
        self.color_jitter = transforms.ColorJitter(**AUG_COLOR_JITTER)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        img  = Image.open(os.path.join(self.image_dir,
                                       self.image_files[index])).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir,
                                       self.mask_files[index])).convert('L')

        # Base resize
        img  = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        if self.augment:
            # Pre-draw all random decisions for synchronized application to img+mask
            do_hflip = random.random() < 0.5
            do_vflip = random.random() < 0.5
            angle    = random.choice([0, 90, 180, 270])
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                img, scale=AUG_CROP_SCALE, ratio=(0.9, 1.1))

            # Apply to image
            if do_hflip: img  = TF.hflip(img)
            if do_vflip: img  = TF.vflip(img)
            if angle:    img  = TF.rotate(img, angle)
            img = TF.resized_crop(img, i, j, h, w,
                                  (self.image_size, self.image_size), Image.BILINEAR)

            # Apply identical geometric transforms to mask
            if do_hflip: mask = TF.hflip(mask)
            if do_vflip: mask = TF.vflip(mask)
            if angle:    mask = TF.rotate(mask, angle)
            mask = TF.resized_crop(mask, i, j, h, w,
                                   (self.image_size, self.image_size), Image.NEAREST)

            # Color jitter — image only (mask is binary geometry, not color)
            img = self.color_jitter(img)

        # Convert to tensor
        img_t  = TF.to_tensor(img)   # [3, H, W] in [0, 1]
        mask_t = TF.to_tensor(mask)  # [1, H, W] in [0, 1]

        # Normalize image to [-1, 1] for DDPM (IMPORTANT: different from GAN-A's [0,1])
        img_t = img_t * 2.0 - 1.0

        # Hard-threshold mask to {0, 1} to remove any NEAREST interpolation artifacts
        mask_t = (mask_t > 0.5).float()

        return img_t, mask_t, self.image_files[index]


# Build datasets
train_ds = LeafDiffusionDataset(
    DISEASED_DIR, LEAF_MASK_DIR, IMAGE_SIZE,
    split='train', train_fraction=TRAIN_SPLIT, seed=SPLIT_SEED, augment=True)
test_ds  = LeafDiffusionDataset(
    DISEASED_DIR, LEAF_MASK_DIR, IMAGE_SIZE,
    split='test',  train_fraction=TRAIN_SPLIT, seed=SPLIT_SEED, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} samples  |  Test: {len(test_ds)} samples')
print(f'Batches per epoch: {len(train_loader)}')
# Verify normalization
sample_img, sample_mask, _ = train_ds[0]
print(f'Image range: [{sample_img.min():.3f}, {sample_img.max():.3f}]  (expected ~[-1, 1])')
print(f'Mask unique values: {sample_mask.unique().tolist()}  (expected [0.0, 1.0])')

Train: 236 samples  |  Test: 27 samples
Batches per epoch: 14
Image range: [-0.961, 0.647]  (expected ~[-1, 1])
Mask unique values: [0.0, 1.0]  (expected [0.0, 1.0])


: 

In [ ]:
# 7. Model, noise schedule, warm-start, optimizer

# --- Noise schedule ---
sched = build_schedule(T_STEPS, SCHEDULE, device)
print(f'Schedule: {SCHEDULE}, T={T_STEPS}')
print(f'  beta  range: [{sched["betas"].min():.5f}, {sched["betas"].max():.5f}]')
print(f'  alpha range: [{sched["alphas_cumprod"].min():.5f}, '
      f'{sched["alphas_cumprod"].max():.5f}]')

# --- Model ---
model = TimeEmbeddedUNet(
    in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS,
    features=FEATURES, time_embed_dim=TIME_EMBED_DIM).to(device)

# --- Training state ---
START_EPOCH = 1
history = {'loss': [], 'loss_t_bucket': [], 'lr': [], 'epoch_time': []}

if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    # Resume full DDPM training state
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    START_EPOCH = ckpt['epoch'] + 1
    if 'history' in ckpt:
        history = ckpt['history']
    print(f"Resumed from epoch {ckpt['epoch']} → continuing from {START_EPOCH}")
    del ckpt
    torch.cuda.empty_cache()

elif WARMSTART_CHECKPOINT and os.path.exists(WARMSTART_CHECKPOINT):
    # Warm-start: load all matching keys from GAN-A Phase-1 checkpoint.
    # Matching keys: downs.*.conv.{0,1,3,4}.*, bottleneck.conv.*.*, ups.*.conv.{0,1,3,4}.*,
    #                ups.*_even.*, final.*
    # Skipped keys (strict=False): time_embed.*, *.time_proj.*
    ckpt         = torch.load(WARMSTART_CHECKPOINT, map_location=device, weights_only=False)
    phase1_state = ckpt.get('netG_state_dict', ckpt)
    model_state  = model.state_dict()

    n_loaded = n_skipped = 0
    for k, v in phase1_state.items():
        if k in model_state and model_state[k].shape == v.shape:
            model_state[k] = v
            n_loaded += 1
        else:
            n_skipped += 1
    model.load_state_dict(model_state, strict=False)
    print(f'Warm-started from {os.path.basename(WARMSTART_CHECKPOINT)}')
    print(f'  Loaded  {n_loaded} tensors  (conv/BN encoder-decoder-bottleneck weights)')
    print(f'  Skipped {n_skipped} tensors (FiLM/time_embed layers — randomly initialized)')
    del ckpt, phase1_state, model_state
    torch.cuda.empty_cache()

else:
    print('No warm-start checkpoint found — training from scratch.')

# --- Optimizer + LR scheduler ---
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = GradScaler('cuda')

# --- Model size ---
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTimeEmbeddedUNet trainable params: {n_params:,}')

# --- Forward-pass sanity check ---
with torch.no_grad():
    dummy_x = torch.randn(1, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=device)
    dummy_t = torch.randint(0, T_STEPS, (1,), device=device)
    dummy_y = model(dummy_x, dummy_t)
print(f'Forward pass OK: input {tuple(dummy_x.shape)} → output {tuple(dummy_y.shape)}')
assert dummy_y.shape == (1, OUT_CHANNELS, IMAGE_SIZE, IMAGE_SIZE), 'Shape mismatch!'
del dummy_x, dummy_t, dummy_y
torch.cuda.empty_cache()

In [ ]:
# 8. DDIM sampler — fast deterministic inference

@torch.no_grad()
def ddim_sample(model, mask, sched, T, ddim_steps=50, eta=0.0, device='cuda'):
    """Generate diseased leaf images from binary leaf masks using DDIM.

    Args:
        model:      trained TimeEmbeddedUNet
        mask:       [B, 1, H, W] binary leaf mask in {0, 1} on device
        sched:      noise schedule dict from build_schedule()
        T:          total DDPM timesteps used during training (e.g. 1000)
        ddim_steps: DDIM denoising steps (e.g. 50)
        eta:        stochasticity — 0.0 = deterministic DDIM; 1.0 = DDPM
        device:     torch device

    Returns:
        [B, 3, H, W] generated image tensor in [-1, 1]
    """
    B, _, H, W = mask.shape

    # Select evenly-spaced DDIM timestep subsequence (reversed for denoising)
    step_size = T // ddim_steps
    timesteps = list(reversed(range(0, T, step_size)))[:ddim_steps]

    # Start from pure Gaussian noise
    x = torch.randn(B, 3, H, W, device=device)

    model.eval()
    for idx, t_val in enumerate(timesteps):
        t_tensor = torch.full((B,), t_val, device=device, dtype=torch.long)
        t_prev   = timesteps[idx + 1] if idx + 1 < len(timesteps) else -1

        # Predict noise: concat noisy image with (static) conditioning mask
        model_in = torch.cat([x, mask], dim=1)  # [B, 4, H, W]
        eps      = model(model_in, t_tensor)    # [B, 3, H, W]

        alpha_t    = sched['alphas_cumprod'][t_val]
        alpha_prev = (sched['alphas_cumprod'][t_prev]
                      if t_prev >= 0 else torch.tensor(1.0, device=device))

        # Predict x0 from current x_t and epsilon (DDIM Eq. 9)
        x0_pred = (x - (1.0 - alpha_t).sqrt() * eps) / alpha_t.sqrt()
        x0_pred = x0_pred.clamp(-1.0, 1.0)

        # DDIM update (Song et al. 2020, Eq. 12)
        sigma   = (eta * ((1.0 - alpha_prev) / (1.0 - alpha_t)
                          * (1.0 - alpha_t / alpha_prev)).clamp(min=0.0).sqrt())
        dir_xt  = (1.0 - alpha_prev - sigma ** 2).clamp(min=0.0).sqrt() * eps
        noise   = sigma * torch.randn_like(x) if eta > 0.0 else 0.0
        x       = alpha_prev.sqrt() * x0_pred + dir_xt + noise

    return x.clamp(-1.0, 1.0)


print('DDIM sampler defined.')
print(f'  Default: {DDIM_STEPS} steps, eta={DDIM_ETA} '
      f'({"deterministic" if DDIM_ETA == 0.0 else "stochastic"})')

In [ ]:
# 9. Training loop — DDPM epsilon-prediction

def save_diffusion_samples(model, sched, test_dataset, device, epoch, sample_dir,
                           ddim_steps=50, eta=0.0, num_samples=3):
    """Generate samples from test masks and save a visual grid:
       [binary mask | DDIM generated | nearest real GLS by L1]
    """
    model.eval()
    n   = min(num_samples, len(test_dataset))
    fig, axs = plt.subplots(n, 3, figsize=(15, 5 * n))
    if n == 1:
        axs = axs[np.newaxis, :]

    col_titles = ['Binary Leaf Mask', 'Generated (DDIM)', 'Nearest Real GLS (L1)']
    for j, title in enumerate(col_titles):
        axs[0, j].set_title(title, fontsize=11, fontweight='bold')

    # Cache all test images for nearest-neighbour lookup (L1 in pixel space)
    real_imgs = [test_dataset[i][0] for i in range(len(test_dataset))]

    def to_np(t):
        """[-1,1] tensor → [0,1] numpy HWC image."""
        return ((t.clamp(-1, 1) + 1) / 2).cpu().permute(1, 2, 0).numpy()

    for i in range(n):
        _, mask_t, fname = test_dataset[i]
        mask_in = mask_t.unsqueeze(0).to(device)

        with torch.no_grad():
            gen = ddim_sample(model, mask_in, sched, T_STEPS, ddim_steps, eta, device)

        gen_cpu = gen.squeeze(0).cpu()
        # Nearest real GLS by L1 distance (sanity check — should NOT be identical)
        dists   = [torch.mean(torch.abs(gen_cpu - r)).item() for r in real_imgs]
        nearest = real_imgs[int(np.argmin(dists))]

        axs[i, 0].imshow(mask_t.squeeze(0).cpu().numpy(), cmap='gray')
        axs[i, 0].axis('off'); axs[i, 0].set_ylabel(fname[:20], fontsize=7)
        axs[i, 1].imshow(to_np(gen_cpu)); axs[i, 1].axis('off')
        axs[i, 2].imshow(to_np(nearest)); axs[i, 2].axis('off')
        del gen, gen_cpu

    plt.suptitle(f'Diffusion Samples — Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(sample_dir, f'samples_epoch{epoch:04d}.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    model.train()


print(f'Training from epoch {START_EPOCH} to {NUM_EPOCHS}')
print('=' * 80)

for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    epoch_start = time.time()
    model.train()

    run_loss     = 0.0
    bucket_loss  = torch.zeros(10, device=device)   # per-timestep-bucket MSE
    bucket_count = torch.zeros(10, device=device)
    n_samples    = 0

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for clean_img, mask, _ in progress:
        B = clean_img.size(0)
        clean_img = clean_img.to(device, non_blocking=True)  # [B, 3, H, W] in [-1, 1]
        mask      = mask.to(device, non_blocking=True)        # [B, 1, H, W] in {0, 1}

        # Sample random timesteps and noise
        t     = torch.randint(0, T_STEPS, (B,), device=device)
        noise = torch.randn_like(clean_img)

        # Forward diffusion: corrupt the image
        x_t, _ = q_sample(clean_img, t, sched, noise)

        # 4-channel input: noisy image + conditioning mask (mask is NEVER noised)
        model_input = torch.cat([x_t, mask], dim=1)  # [B, 4, H, W]

        optimizer.zero_grad()
        with autocast('cuda'):
            pred_noise = model(model_input, t)        # [B, 3, H, W]
            loss       = F.mse_loss(pred_noise, noise)

        scaler.scale(loss).backward()
        # Gradient clipping — essential for DDPM; prevents spikes at high-noise timesteps
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        run_loss  += loss.item() * B
        n_samples += B

        # Per-bucket loss tracking (10 buckets of 100 timesteps each)
        buckets = (t * 10 // T_STEPS).clamp(0, 9)
        with torch.no_grad():
            for b in range(10):
                m = buckets == b
                if m.any():
                    bucket_loss[b]  += F.mse_loss(pred_noise[m], noise[m]) * m.sum()
                    bucket_count[b] += m.sum()

        progress.set_postfix({'MSE': f'{loss.item():.5f}'})

    scheduler.step()

    avg_loss   = run_loss / n_samples
    epoch_time = time.time() - epoch_start
    lr_now     = scheduler.get_last_lr()[0]
    bucket_avg = (bucket_loss / bucket_count.clamp(min=1)).cpu().tolist()

    history['loss'].append(avg_loss)
    history['loss_t_bucket'].append(bucket_avg)
    history['lr'].append(lr_now)
    history['epoch_time'].append(epoch_time)

    print(f'Epoch {epoch}/{NUM_EPOCHS} [{epoch_time:.1f}s | lr={lr_now:.2e}] | '
          f'MSE loss: {avg_loss:.6f}')

    # --- Save checkpoint ---
    if epoch % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'diffusion_epoch{epoch}.pth')
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'optim_state_dict': optimizer.state_dict(),
            'sched_state_dict': scheduler.state_dict(),
            'loss':             avg_loss,
            'history':          history,
        }, ckpt_path)
        print(f'  >> Checkpoint saved: {ckpt_path}')

    # --- Save visual samples ---
    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        save_diffusion_samples(model, sched, test_ds, device, epoch, SAMPLE_DIR,
                               ddim_steps=DDIM_STEPS, eta=DDIM_ETA)
        print(f'  >> Samples saved to {SAMPLE_DIR}')

    torch.cuda.empty_cache()

# --- Save final model ---
final_path = os.path.join(CHECKPOINT_DIR, f'diffusion_epoch{NUM_EPOCHS}_final.pth')
torch.save({
    'epoch':            NUM_EPOCHS,
    'model_state_dict': model.state_dict(),
    'loss':             history['loss'][-1],
    'history':          history,
}, final_path)
print(f'Final model saved: {final_path}')

log_path = os.path.join(LOG_DIR, 'diffusion_history.json')
with open(log_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved: {log_path}')

del optimizer, scaler
torch.cuda.empty_cache()
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# 10. Training metrics plots

epochs = range(1, len(history['loss']) + 1)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# MSE loss over training (log scale)
axs[0].plot(epochs, history['loss'], 'b-', linewidth=1.5)
axs[0].set_title('DDPM MSE Loss (ε-prediction)')
axs[0].set_xlabel('Epoch'); axs[0].set_ylabel('MSE'); axs[0].grid(True)
axs[0].set_yscale('log')

# Learning rate schedule
axs[1].plot(epochs, history['lr'], 'g-', linewidth=1.5)
axs[1].set_title('Learning Rate (CosineAnnealingLR)')
axs[1].set_xlabel('Epoch'); axs[1].set_ylabel('LR'); axs[1].grid(True)

# Per-timestep-bucket loss at final epoch
if history['loss_t_bucket']:
    bucket_labels  = [f'{i * 100}–{(i + 1) * 100}' for i in range(10)]
    final_buckets  = history['loss_t_bucket'][-1]
    axs[2].bar(range(10), final_buckets, color='purple', alpha=0.7)
    axs[2].set_xticks(range(10))
    axs[2].set_xticklabels(bucket_labels, rotation=45, ha='right', fontsize=8)
    axs[2].set_title(f'Loss by Timestep Bucket (epoch {len(history["loss"])})')
    axs[2].set_xlabel('Timestep range (0=low noise, 9=high noise)')
    axs[2].set_ylabel('MSE'); axs[2].grid(True, axis='y')

plt.suptitle('Diffusion Model Training Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(LOG_DIR, 'diffusion_loss_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Loss curves saved: {fig_path}')

total_time = sum(history['epoch_time'])
print(f'\nFinal MSE loss:      {history["loss"][-1]:.6f}')
print(f'Total training time: {total_time / 3600:.2f} hours')

## Inference

Cells below run DDIM on test masks (or any custom binary mask).
Load any saved checkpoint before running.

In [ ]:
# 11. Load checkpoint for inference
INFER_CHECKPOINT = os.path.join(CHECKPOINT_DIR, f'diffusion_epoch{NUM_EPOCHS}_final.pth')

ckpt = torch.load(INFER_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  (loss: {ckpt['loss']:.6f})")
del ckpt

# List test images
print(f'\nTest set: {len(test_ds)} images')
for i in range(len(test_ds)):
    _, _, fname = test_ds[i]
    print(f'  [{i:2d}] {fname}')

In [ ]:
# 12. Visualise single test sample — G: mask → generated diseased leaf
TEST_INDEX = 0   # <-- change this

torch.cuda.empty_cache()
model.eval()
_, mask_t, fname = test_ds[TEST_INDEX]
mask_in = mask_t.unsqueeze(0).to(device)

with torch.no_grad():
    gen_det  = ddim_sample(model, mask_in, sched, T_STEPS, DDIM_STEPS, eta=0.0, device=device)
    gen_stoc = ddim_sample(model, mask_in, sched, T_STEPS, DDIM_STEPS, eta=1.0, device=device)

def t2np(t):
    """Convert [-1,1] CHW tensor to [0,1] HWC numpy."""
    return ((t.squeeze(0).clamp(-1, 1) + 1) / 2).cpu().permute(1, 2, 0).numpy()

fig, axs = plt.subplots(1, 4, figsize=(24, 5))
axs[0].imshow(mask_t.squeeze(0).cpu().numpy(), cmap='gray')
axs[0].set_title(f'Input Mask\n{fname}'); axs[0].axis('off')
axs[1].imshow(t2np(gen_det))
axs[1].set_title(f'Generated (η=0.0, det.)\nDDIM {DDIM_STEPS} steps'); axs[1].axis('off')
axs[2].imshow(t2np(gen_stoc))
axs[2].set_title(f'Generated (η=1.0, stoc.)\nDDIM {DDIM_STEPS} steps'); axs[2].axis('off')
# Show the original diseased image for reference
real_t, _, _ = test_ds[TEST_INDEX]
axs[3].imshow(t2np(real_t))
axs[3].set_title('Real GLS Image (reference)'); axs[3].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, f'infer_test{TEST_INDEX}.png'),
            dpi=150, bbox_inches='tight')
plt.show()
del gen_det, gen_stoc

In [ ]:
# 13. Diversity evaluation — N seeds per test mask
# High L1 between samples = model is diverse (not collapsed).
# Target: mean pairwise L1 > 0.05.

torch.cuda.empty_cache()
model.eval()

N_SEEDS = 5
N_EVAL  = min(10, len(test_ds))

print(f'Evaluating {N_EVAL} test masks x {N_SEEDS} seeds each (eta=1.0 stochastic)...')
diversity_l1 = []

fig, axs = plt.subplots(N_EVAL, N_SEEDS + 1, figsize=(4 * (N_SEEDS + 1), 4 * N_EVAL))
if N_EVAL == 1:
    axs = axs[np.newaxis, :]

for i in range(N_EVAL):
    _, mask_t, fname = test_ds[i]
    mask_in = mask_t.unsqueeze(0).to(device)
    samples = []

    with torch.no_grad():
        for s in range(N_SEEDS):
            torch.manual_seed(s)
            gen = ddim_sample(model, mask_in, sched, T_STEPS, DDIM_STEPS,
                              eta=1.0, device=device)
            samples.append(gen.squeeze(0).cpu())
            del gen

    # Pairwise L1 diversity
    pairs = []
    for a in range(N_SEEDS):
        for b in range(a + 1, N_SEEDS):
            pairs.append(torch.mean(torch.abs(samples[a] - samples[b])).item())
    diversity_l1.append(float(np.mean(pairs)))

    axs[i, 0].imshow(mask_t.squeeze(0).numpy(), cmap='gray')
    axs[i, 0].set_title(f'Mask\n{fname[:15]}', fontsize=7)
    axs[i, 0].axis('off')
    for s, samp in enumerate(samples):
        img_np = ((samp.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy()
        axs[i, s + 1].imshow(img_np)
        axs[i, s + 1].set_title(f'Seed {s}', fontsize=7)
        axs[i, s + 1].axis('off')

plt.suptitle(
    f'Intra-mask Diversity (η=1.0)  |  Mean pairwise L1: {np.mean(diversity_l1):.4f}',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'eval_diversity.png'), dpi=100, bbox_inches='tight')
plt.show()

print('=' * 55)
print('DIVERSITY EVALUATION')
print('=' * 55)
for i, d in enumerate(diversity_l1):
    _, _, fname = test_ds[i]
    print(f'  [{i:2d}] {fname[:25]:25s}: mean pairwise L1 = {d:.4f}')
print(f'  Overall mean L1: {np.mean(diversity_l1):.4f}  (>0.05 = not collapsed)')
print('=' * 55)

In [ ]:
# 14. Inference on a custom binary mask
# Change CUSTOM_MASK_PATH to any binary PNG (white = leaf, black = background).

CUSTOM_MASK_PATH = os.path.join(
    str(repo_root), 'data', 'masks', 'leaf', 'gls', 'leaf_mask_001.png')  # <-- change this

torch.cuda.empty_cache()
model.eval()

mask_img = Image.open(CUSTOM_MASK_PATH).convert('L')
mask_img = mask_img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
mask_t   = (TF.to_tensor(mask_img) > 0.5).float().unsqueeze(0).to(device)

with torch.no_grad():
    gen_det  = ddim_sample(model, mask_t, sched, T_STEPS, DDIM_STEPS, eta=0.0, device=device)
    gen_stoc = ddim_sample(model, mask_t, sched, T_STEPS, DDIM_STEPS, eta=1.0, device=device)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(mask_t.squeeze(0).squeeze(0).cpu().numpy(), cmap='gray')
axs[0].set_title(f'Input Mask\n{os.path.basename(CUSTOM_MASK_PATH)}')
axs[0].axis('off')
axs[1].imshow(t2np(gen_det))
axs[1].set_title('Generated — deterministic (η=0.0)'); axs[1].axis('off')
axs[2].imshow(t2np(gen_stoc))
axs[2].set_title('Generated — stochastic (η=1.0)'); axs[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'inference_custom.png'), dpi=150, bbox_inches='tight')
plt.show()
del gen_det, gen_stoc